<a href="https://colab.research.google.com/github/yifeica0/ECS171-Group8/blob/models/ModelTraining/FE_ML/with_sw/text_logistic_NB_RF_with_sw_AND_feature_VADER_POS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# unable colab to access git
!git clone https://github.com/yifeica0/ECS171-Group8.git
%cd ECS171-Group8
!git checkout models

Cloning into 'ECS171-Group8'...
remote: Enumerating objects: 843, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 843 (delta 76), reused 85 (delta 30), pack-reused 679 (from 1)
Receiving objects: 100% (843/843), 204.77 MiB | 16.81 MiB/s, done.
Resolving deltas: 100% (376/376), done.
/content/ECS171-Group8
Branch 'models' set up to track remote branch 'models' from 'origin'.
Switched to a new branch 'models'


In [5]:
# update repo informations
!git pull

Already up to date.


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, RocCurveDisplay
from sklearn.preprocessing import LabelBinarizer
from scipy.sparse import hstack

In [ ]:
import nltk

# Download necessary NLTK data for VADER and POS tagging
# This might take a moment if not already downloaded
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')
try:
    nltk.data.find('taggers/averaged_perceptron_tagger.zip')
except LookupError:
    nltk.download('averaged_perceptron_tagger')
try:
    nltk.data.find('taggers/averaged_perceptron_tagger_eng.zip')
except LookupError:
    nltk.download('averaged_perceptron_tagger_eng')

# Download required NLTK data (run once)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [7]:
df = pd.read_parquet("datasets/with_stop_word/amazon_user_reviews_textANDfeature_with_sw_final.parquet")
df.head()

,original_text,sentiment,exclamation_count,question_count,word_count,char_count,all_caps_words,uppercase_ratio,total_punctuation,avg_word_length,...,vader_neu,vader_pos,vader_compound,NN_count,JJ_count,VB_count,RB_count,DT_count,PRP_count,PUNCT_count
0,The material isn’t what I expected. I thought ...,2,0,0,14,73,0,0.041096,2,5.214286,...,1.000,0.000,0.0000,5,0,4,0,2,3,2
1,the magnetic snap tore out of the fake leather...,2,0,0,14,70,0,0.000000,1,5.000000,...,0.795,0.000,-0.4767,5,2,1,0,3,0,1
2,[[VIDEOID:f43161a0116bd3fb943546242e053d92]] T...,2,1,0,44,293,1,0.044369,5,6.659091,...,0.798,0.202,0.8718,26,4,10,5,5,3,5
3,Would not recommend,2,0,0,3,19,0,0.052632,0,6.333333,...,0.487,0.000,-0.2755,0,0,1,1,0,0,0
4,"they are all too small, but very pretty",2,0,0,8,39,0,0.000000,1,4.875000,...,0.596,0.404,0.6946,0,1,1,3,1,1,1


In [8]:
import pandas as pd
from scipy.sparse import hstack

# Drop rows where 'original_text' is NaN or empty string
df.dropna(subset=['original_text'], inplace=True)
df = df[df['original_text'].str.strip() != '']

# Split data into features (X) and target (y)
X_all_features = df.drop(['sentiment'], axis=1)
y = df['sentiment']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_all_features, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training data size: {X_train.shape[0]} samples")
print(f"Testing data size: {X_test.shape[0]} samples")

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Fit and transform the 'original_text' column for training and testing data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train['original_text'])
X_test_tfidf = tfidf_vectorizer.transform(X_test['original_text'])

# Drop the 'original_text' column from X_train and X_test to concatenate with TF-IDF features
X_train_numerical = X_train.drop(columns=['original_text'])
X_test_numerical = X_test.drop(columns=['original_text'])

# Ensure 'vader_compound' is non-negative for Multinomial Naive Bayes
# If 'vader_compound' is present and has negative values, shift it to a non-negative range
if 'vader_compound' in X_train_numerical.columns:
    min_vader_compound_train = X_train_numerical['vader_compound'].min()
    if min_vader_compound_train < 0:
        X_train_numerical['vader_compound'] = X_train_numerical['vader_compound'] + abs(min_vader_compound_train)
        X_test_numerical['vader_compound'] = X_test_numerical['vader_compound'] + abs(min_vader_compound_train)

# Convert numerical features to sparse matrix format if they are not already (hstack requires sparse or numpy arrays)
X_train_numerical_sparse = X_train_numerical.sparse.to_coo() if pd.api.types.is_sparse(X_train_numerical) else X_train_numerical.values
X_test_numerical_sparse = X_test_numerical.sparse.to_coo() if pd.api.types.is_sparse(X_test_numerical) else X_test_numerical.values

# Combine TF-IDF features with other numerical features
X_train_combined = hstack([X_train_tfidf, X_train_numerical_sparse])
X_test_combined = hstack([X_test_tfidf, X_test_numerical_sparse])

print(f"TF-IDF vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"Shape of X_train_combined: {X_train_combined.shape}")
print(f"Shape of X_test_combined: {X_test_combined.shape}")

Training data size: 14720 samples
Testing data size: 3680 samples
TF-IDF vocabulary size: 5000
Shape of X_train_combined: (14720, 5022)
Shape of X_test_combined: (3680, 5022)


/tmp/ipykernel_12883/1419133197.py:38: DeprecationWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  X_train_numerical_sparse = X_train_numerical.sparse.to_coo() if pd.api.types.is_sparse(X_train_numerical) else X_train_numerical.values
/tmp/ipykernel_12883/1419133197.py:39: DeprecationWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  X_test_numerical_sparse = X_test_numerical.sparse.to_coo() if pd.api.types.is_sparse(X_test_numerical) else X_test_numerical.values


### Model Training and Evaluation

#### Logistic Regression

In [ ]:
# Initialize and train Logistic Regression model
lr_model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)
lr_model.fit(X_train_combined, y_train)

# Predict on the test set
y_pred_lr = lr_model.predict(X_test_combined)

# Evaluate the model
print("Logistic Regression Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr, digits=4))

Logistic Regression Performance:
Accuracy: 0.7098
              precision    recall  f1-score   support

           0     0.7808    0.8099    0.7951      1236
           1     0.6383    0.5801    0.6078      1217
           2     0.7005    0.7376    0.7185      1227

    accuracy                         0.7098      3680
   macro avg     0.7065    0.7092    0.7071      3680
weighted avg     0.7069    0.7098    0.7076      3680



#### Multinomial Naive Bayes


In [ ]:
from sklearn.preprocessing import MaxAbsScaler
from sklearn.naive_bayes import MultinomialNB

# Initialize and train Multinomial Naive Bayes model
mnb_model = MultinomialNB()

# Scale combined features to a non-negative range for MNB
scaler = MaxAbsScaler()
X_train_scaled = scaler.fit_transform(X_train_combined)
X_test_scaled = scaler.transform(X_test_combined)

mnb_model.fit(X_train_scaled, y_train)

# Predict on the test set using scaled features
y_pred_mnb = mnb_model.predict(X_test_scaled)

# Evaluate the model
print("Multinomial Naive Bayes Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_mnb):.4f}")
print(classification_report(y_test, y_pred_mnb, digits=4))

Multinomial Naive Bayes Performance:
Accuracy: 0.6799
              precision    recall  f1-score   support

           0     0.7697    0.7435    0.7564      1236
           1     0.5808    0.6056    0.5929      1217
           2     0.6952    0.6895    0.6923      1227

    accuracy                         0.6799      3680
   macro avg     0.6819    0.6795    0.6805      3680
weighted avg     0.6824    0.6799    0.6810      3680



#### Support Vector Machine (SVM)

#### Non-linear Support Vector Machine (SVC)

In [9]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Re-define X_all_features and y to ensure this cell is self-contained
X_all_features_svm = df.drop(['sentiment'], axis=1)
y_svm = df['sentiment']

# Split the dataset into training and testing sets
X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(X_all_features_svm, y_svm, test_size=0.2, random_state=42, stratify=y_svm)

# Drop the 'original_text' column from X_train_svm and X_test_svm to use only numerical features
X_train_numerical_svm = X_train_svm.drop(columns=['original_text'])
X_test_numerical_svm = X_test_svm.drop(columns=['original_text'])

# Convert numerical features to numpy arrays (SVC expects dense arrays for most operations, especially with non-linear kernels)
X_train_numerical_dense_svm = X_train_numerical_svm.values
X_test_numerical_dense_svm = X_test_numerical_svm.values

# Scale numerical features (important for SVC, especially with RBF kernel)
scaler_svm_nonlinear = StandardScaler()
X_train_scaled_svm_nonlinear = scaler_svm_nonlinear.fit_transform(X_train_numerical_dense_svm)
X_test_scaled_svm_nonlinear = scaler_svm_nonlinear.transform(X_test_numerical_dense_svm)

# Initialize and train SVC model (non-linear with RBF kernel)
# Set probability=True if you intend to plot ROC curves later (adds computational cost)
svm_nonlinear_model = SVC(kernel='rbf', random_state=42, probability=True)
svm_nonlinear_model.fit(X_train_scaled_svm_nonlinear, y_train_svm)

# Predict on the test set
y_pred_svm_nonlinear = svm_nonlinear_model.predict(X_test_scaled_svm_nonlinear)

# Evaluate the model
print("Support Vector Machine (Non-linear SVC) Performance:")
print(f"Accuracy: {accuracy_score(y_test_svm, y_pred_svm_nonlinear):.4f}")
print(classification_report(y_test_svm, y_pred_svm_nonlinear, digits=4))

Support Vector Machine (Non-linear SVC) Performance:
Accuracy: 0.6008
              precision    recall  f1-score   support

           0     0.6889    0.7023    0.6955      1236
           1     0.5151    0.4199    0.4627      1217
           2     0.5826    0.6781    0.6267      1227

    accuracy                         0.6008      3680
   macro avg     0.5955    0.6001    0.5950      3680
weighted avg     0.5960    0.6008    0.5956      3680



In [ ]:
from sklearn.svm import LinearSVC

# Initialize and train LinearSVC model (more efficient for large linear SVMs)
# Using LinearSVC instead of SVC(kernel='linear') for better performance on large datasets
svm_model = LinearSVC(random_state=42, max_iter=1000, dual=False) # dual=False is recommended for n_samples > n_features
svm_model.fit(X_train_combined, y_train)

# Predict on the test set
y_pred_svm = svm_model.predict(X_test_combined)

# Evaluate the model
print("Support Vector Machine (LinearSVC) Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print(classification_report(y_test, y_pred_svm, digits=4))

#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train Random Forest model
# n_estimators is the number of trees in the forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_combined, y_train)

# Predict on the test set
y_pred_rf = rf_model.predict(X_test_combined)

# Evaluate the model
print("Random Forest Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(classification_report(y_test, y_pred_rf, digits=4))

Random Forest Performance:
Accuracy: 0.6606
              precision    recall  f1-score   support

           0     0.7463    0.7379    0.7421      1236
           1     0.5772    0.5440    0.5601      1217
           2     0.6537    0.6985    0.6753      1227

    accuracy                         0.6606      3680
   macro avg     0.6591    0.6601    0.6592      3680
weighted avg     0.6595    0.6606    0.6596      3680



In [10]:
# Initialize and train SVC model (non-linear with RBF kernel)
# Set probability=True if you intend to plot ROC curves later (adds computational cost)
rf_model_no_tfidf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model_no_tfidf.fit(X_train_scaled_svm_nonlinear, y_train_svm)

# Predict on the test set
y_pred__no_tfidf = rf_model_no_tfidf.predict(X_test_scaled_svm_nonlinear)

# Evaluate the model
print("no tfidf Random Forest Performance:")
print(f"Accuracy: {accuracy_score(y_test_svm, y_pred__no_tfidf):.4f}")
print(classification_report(y_test_svm, y_pred__no_tfidf, digits=4))

no tfidf Random Forest Performance:
Accuracy: 0.5861
              precision    recall  f1-score   support

           0     0.6869    0.6958    0.6913      1236
           1     0.4833    0.4289    0.4545      1217
           2     0.5749    0.6316    0.6019      1227

    accuracy                         0.5861      3680
   macro avg     0.5817    0.5854    0.5826      3680
weighted avg     0.5822    0.5861    0.5832      3680



### Confusion Matrices visualization

In [ ]:
models = {
    "Logistic Regression": (lr_model, y_pred_lr),
    "Multinomial Naive Bayes": (mnb_model, y_pred_mnb),
    "SVM": (svm_model, y_pred_svm),
    "Random Forest": (rf_model, y_pred_rf)
}

for name, (model, y_pred) in models.items():
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Neutral', 'Positive'],
                yticklabels=['Negative', 'Neutral', 'Positive'])
    plt.title(f'Confusion Matrix for {name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

### ROC Curves Visualization

In [ ]:
lb = LabelBinarizer()
# y_test_binarized will have columns for each class (0, 1, 2)
y_test_binarized = lb.fit_transform(y_test)


plt.figure(figsize=(15, 10))

# --- Logistic Regression ROC ---
plt.subplot(2, 2, 1)
y_score_lr = lr_model.predict_proba(X_test_combined)
RocCurveDisplay.from_predictions(y_test_binarized[:, 0], y_score_lr[:, 0], name=f'Class 0 (AUC={auc(roc_curve(y_test_binarized[:, 0], y_score_lr[:, 0])[0], roc_curve(y_test_binarized[:, 0], y_score_lr[:, 0])[1]):.2f})', pos_label=1, ax=plt.gca())
RocCurveDisplay.from_predictions(y_test_binarized[:, 1], y_score_lr[:, 1], name=f'Class 1 (AUC={auc(roc_curve(y_test_binarized[:, 1], y_score_lr[:, 1])[0], roc_curve(y_test_binarized[:, 1], y_score_lr[:, 1])[1]):.2f})', pos_label=1, ax=plt.gca())
RocCurveDisplay.from_predictions(y_test_binarized[:, 2], y_score_lr[:, 2], name=f'Class 2 (AUC={auc(roc_curve(y_test_binarized[:, 2], y_score_lr[:, 2])[0], roc_curve(y_test_binarized[:, 2], y_score_lr[:, 2])[1]):.2f})', pos_label=1, ax=plt.gca())
plt.title('ROC Curve for Logistic Regression')
plt.plot([0, 1], [0, 1], 'k--', label='Chance Level')
plt.legend()


# --- Multinomial Naive Bayes ROC ---
plt.subplot(2, 2, 2)
# Predict probabilities using the scaled test data
y_score_mnb = mnb_model.predict_proba(X_test_scaled)
RocCurveDisplay.from_predictions(y_test_binarized[:, 0], y_score_mnb[:, 0], name=f'Class 0 (AUC={auc(roc_curve(y_test_binarized[:, 0], y_score_mnb[:, 0])[0], roc_curve(y_test_binarized[:, 0], y_score_mnb[:, 0])[1]):.2f})', pos_label=1, ax=plt.gca())
RocCurveDisplay.from_predictions(y_test_binarized[:, 1], y_score_mnb[:, 1], name=f'Class 1 (AUC={auc(roc_curve(y_test_binarized[:, 1], y_score_mnb[:, 1])[0], roc_curve(y_test_binarized[:, 1], y_score_mnb[:, 1])[1]):.2f})', pos_label=1, ax=plt.gca())
RocCurveDisplay.from_predictions(y_test_binarized[:, 2], y_score_mnb[:, 2], name=f'Class 2 (AUC={auc(roc_curve(y_test_binarized[:, 2], y_score_mnb[:, 2])[0], roc_curve(y_test_binarized[:, 2], y_score_mnb[:, 2])[1]):.2f})', pos_label=1, ax=plt.gca())
plt.title('ROC Curve for Multinomial Naive Bayes')
plt.plot([0, 1], [0, 1], 'k--', label='Chance Level')
plt.legend()


# --- SVM (SVC) ROC (Requires probability=True during SVC initialization to get predict_proba) ---
# As SVM was initialized without probability=True, we can't directly use predict_proba.
# If you need ROC for SVM, you would need to re-initialize and retrain it with probability=True.
# For now, we will skip SVM ROC or provide a placeholder.
# If probability=True was set, the code would look like this:
# plt.subplot(2, 2, 3)
# y_score_svm = svm_model.predict_proba(X_test_tfidf)
# RocCurveDisplay.from_predictions(y_test_binarized[:, 0], y_score_svm[:, 0], name=f'Class 0 (AUC={auc(roc_curve(y_test_binarized[:, 0], y_score_svm[:, 0])[0], roc_curve(y_test_binarized[:, 0], y_score_svm[:, 0])[1]):.2f})', pos_label=1, ax=plt.gca())
# ... and so on for other classes
# plt.title('ROC Curve for SVM')
# plt.plot([0, 1], [0, 1], 'k--', label='Chance Level')
# plt.legend()


# --- Random Forest ROC ---
plt.subplot(2, 2, 4)
y_score_rf = rf_model.predict_proba(X_test_combined)
RocCurveDisplay.from_predictions(y_test_binarized[:, 0], y_score_rf[:, 0], name=f'Class 0 (AUC={auc(roc_curve(y_test_binarized[:, 0], y_score_rf[:, 0])[0], roc_curve(y_test_binarized[:, 0], y_score_rf[:, 0])[1]):.2f})', pos_label=1, ax=plt.gca())
RocCurveDisplay.from_predictions(y_test_binarized[:, 1], y_score_rf[:, 1], name=f'Class 1 (AUC={auc(roc_curve(y_test_binarized[:, 1], y_score_rf[:, 1])[0], roc_curve(y_test_binarized[:, 1], y_score_rf[:, 1])[1]):.2f})', pos_label=1, ax=plt.gca())
RocCurveDisplay.from_predictions(y_test_binarized[:, 2], y_score_rf[:, 2], name=f'Class 2 (AUC={auc(roc_curve(y_test_binarized[:, 2], y_score_rf[:, 2])[0], roc_curve(y_test_binarized[:, 2], y_score_rf[:, 2])[1]):.2f})', pos_label=1, ax=plt.gca())
plt.title('ROC Curve for Random Forest')
plt.plot([0, 1], [0, 1], 'k--', label='Chance Level')
plt.legend()

plt.tight_layout()
plt.show()